In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [2]:
!pip install yfinance


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import yfinance as yf

In [4]:
bitcoin = yf.download(tickers = "BTC-USD" , start = "2017-01-01" , end = "2025-12-31", interval="1d")

[*********************100%***********************]  1 of 1 completed


In [5]:
bitcoin

Price,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Date,,,,,
2017-01-01,998.325012,1003.080017,958.698975,963.658020,147775008
2017-01-02,1021.750000,1031.390015,996.702026,998.617004,222184992
2017-01-03,1043.839966,1044.079956,1021.599976,1021.599976,185168000
2017-01-04,1154.729980,1159.420044,1044.400024,1044.400024,344945984
2017-01-05,1013.380005,1191.099976,910.416992,1156.729980,510199008
...,...,...,...,...,...
2025-12-26,87301.429688,89459.429688,86628.140625,87235.507812,42455674908
2025-12-27,87802.156250,87874.781250,87182.976562,87301.429688,13741199310


In [6]:
bitcoin = bitcoin.reset_index()

In [7]:
bitcoin.columns.droplevel(1)

Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='str', name='Price')

In [8]:
bitcoin.columns = bitcoin.columns.droplevel(1)

In [9]:
bitcoin

Price,Date,Close,High,Low,Open,Volume
0,2017-01-01,998.325012,1003.080017,958.698975,963.658020,147775008
1,2017-01-02,1021.750000,1031.390015,996.702026,998.617004,222184992
2,2017-01-03,1043.839966,1044.079956,1021.599976,1021.599976,185168000
3,2017-01-04,1154.729980,1159.420044,1044.400024,1044.400024,344945984
4,2017-01-05,1013.380005,1191.099976,910.416992,1156.729980,510199008
...,...,...,...,...,...,...
3281,2025-12-26,87301.429688,89459.429688,86628.140625,87235.507812,42455674908
3282,2025-12-27,87802.156250,87874.781250,87182.976562,87301.429688,13741199310
3283,2025-12-28,87835.835938,87986.890625,87394.953125,87799.343750,15156557929
3284,2025-12-29,87138.140625,90299.156250,86717.914062,87835.789062,48411625849


In [10]:
bitcoin.columns.name

'Price'

In [11]:
bitcoin.columns.name = None

In [12]:
bitcoin

,Date,Close,High,Low,Open,Volume
0,2017-01-01,998.325012,1003.080017,958.698975,963.658020,147775008
1,2017-01-02,1021.750000,1031.390015,996.702026,998.617004,222184992
2,2017-01-03,1043.839966,1044.079956,1021.599976,1021.599976,185168000
3,2017-01-04,1154.729980,1159.420044,1044.400024,1044.400024,344945984
4,2017-01-05,1013.380005,1191.099976,910.416992,1156.729980,510199008
...,...,...,...,...,...,...
3281,2025-12-26,87301.429688,89459.429688,86628.140625,87235.507812,42455674908
3282,2025-12-27,87802.156250,87874.781250,87182.976562,87301.429688,13741199310
3283,2025-12-28,87835.835938,87986.890625,87394.953125,87799.343750,15156557929
3284,2025-12-29,87138.140625,90299.156250,86717.914062,87835.789062,48411625849


In [13]:
bitcoin.shape

(3286, 6)

In [14]:
bitcoin["Date"].min()

Timestamp('2017-01-01 00:00:00')

In [15]:
bitcoin["Date"].max()

Timestamp('2025-12-30 00:00:00')

In [16]:
bitcoin.dtypes

Date      datetime64[s]
Close           float64
High            float64
Low             float64
Open            float64
Volume            int64
dtype: object

In [17]:
bitcoin.isnull().sum()

Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [18]:
bitcoin.duplicated().sum()

np.int64(0)

In [19]:
bitcoin = bitcoin.sort_values(by = ["Date"])

In [20]:
data = bitcoin.copy()

In [21]:
bitcoin_Sample = data[0:100]

In [22]:
import plotly.graph_objs as go

In [23]:
trace = go.Candlestick( x = bitcoin_Sample["Date"],
                high = bitcoin_Sample["High"],
                open = bitcoin_Sample["Open"],
                close = bitcoin_Sample["Close"],
                low = bitcoin_Sample["Low"]
)

In [24]:
candle_data = [trace]
layout = {
    'title' : "Bitcoin Candlestick Chart",
    'xaxis': {'title' : "Date"}
}

In [25]:
fig = go.Figure(data = candle_data, layout = layout)

In [26]:
%pip install nbformat



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [27]:
import plotly.io as pio

# Bypasses the internal window
pio.renderers.default = "browser"

fig.show()


In [28]:
import plotly.graph_objs as go
import plotly.io as pio

# Force browser view
pio.renderers.default = "browser"

# Step 1: Ensure the Date column is a proper datetime object
data["Date"] = pd.to_datetime(data["Date"])

# Step 2: Temporarily set Date as the index so Pandas can group it by time
df_time = data.set_index("Date")

# Step 3: Resample the data (Choose 'W' for Weekly or 'M' for Monthly)
# We will use 'W' (Weekly) here so it stays detailed but perfectly clean
resampled_df = df_time.resample("W").agg(
    {
        "Open": "first",  # First price of the week
        "High": "max",  # Highest price during the week
        "Low": "min",  # Lowest price during the week
        "Close": "last",  # Final closing price of the week
    }
)

# Step 4: Build the new clean Candlestick chart
trace = go.Candlestick(
    x=resampled_df.index,  # The dates are now the index
    open=resampled_df["Open"],
    high=resampled_df["High"],
    low=resampled_df["Low"],
    close=resampled_df["Close"],
)

layout = {
    "title": "Bitcoin Weekly Trends (Clean Candlestick Chart)",
    "xaxis": {"title": "Timeline"},
    "yaxis": {"title": "Price (USD)"},
}

fig = go.Figure(data=[trace], layout=layout)
fig.show()


In [29]:
data["daily_return"] = data["Close"].pct_change()

In [30]:
print(data.head(2))

        Date        Close         High         Low        Open     Volume  \
0 2017-01-01   998.325012  1003.080017  958.698975  963.658020  147775008   
1 2017-01-02  1021.750000  1031.390015  996.702026  998.617004  222184992   

   daily_return  
0           NaN  
1      0.023464  


In [31]:
data["daily_return"].std()

np.float64(0.03609067393340374)

In [32]:
data["volatility_30d"] = data["daily_return"].rolling(30).std()
data["volatility_90d"] = data["daily_return"].rolling(90).std()

In [33]:
px.line(data,
        x="Date",
        y=["volatility_30d","volatility_90d"],
        title = "Bitcoin Volatility over time",
        labels={
            "value" : "Volatility(risk)",
            "variable" : "window"
        })

In [34]:
data.columns

Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'daily_return',
       'volatility_30d', 'volatility_90d'],
      dtype='str')

In [35]:
print(data[["volatility_30d","volatility_90d"]].describe().T)

                 count      mean       std       min       25%       50%  \
volatility_30d  3256.0  0.032880  0.014350  0.008887  0.022904  0.029760   
volatility_90d  3196.0  0.034026  0.011848  0.014488  0.026066  0.031606   

                     75%      max  
volatility_30d  0.038953  0.09133  
volatility_90d  0.039683  0.07474  


In [36]:
data["cumulative_return"] = (1 + data["daily_return"]).cumprod()

In [37]:
investment = 100000

In [38]:
data["investment_value"] = investment * data["cumulative_return"]

In [39]:
data.columns

Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'daily_return',
       'volatility_30d', 'volatility_90d', 'cumulative_return',
       'investment_value'],
      dtype='str')

In [40]:
print(data[["Date","investment_value"]])

           Date  investment_value
0    2017-01-01               NaN
1    2017-01-02      1.023464e+05
2    2017-01-03      1.045591e+05
3    2017-01-04      1.156667e+05
4    2017-01-05      1.015080e+05
...         ...               ...
3281 2025-12-26      8.744790e+06
3282 2025-12-27      8.794947e+06
3283 2025-12-28      8.798321e+06
3284 2025-12-29      8.728434e+06
3285 2025-12-30      8.857850e+06

[3286 rows x 2 columns]


In [41]:
final_value = data["investment_value"].iloc[-1]

In [42]:
final_value

np.float64(8857850.071992515)

In [43]:
px.line(data,
        x="Date",
        y=["investment_value"],
        title = "1 Lakh Investment Growth In Bitcoin",
        labels={
            "Investmentt_growth" : "Investment Value(Rupees)"
        })

In [44]:
data.columns

Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'daily_return',
       'volatility_30d', 'volatility_90d', 'cumulative_return',
       'investment_value'],
      dtype='str')

In [45]:
print(data.nsmallest(10, "daily_return")[["Date","Close","daily_return"]])

           Date         Close  daily_return
1166 2020-03-12   4970.788086     -0.371695
256  2017-09-14   3154.949951     -0.187411
380  2018-01-16  11490.500000     -0.168548
1989 2022-06-13  22487.388672     -0.159747
400  2018-02-05   6955.270020     -0.159688
2138 2022-11-09  15880.780273     -0.143490
10   2017-01-11    777.757019     -0.143136
907  2019-06-27  11182.806641     -0.140857
1599 2021-05-19  37002.441406     -0.137661
687  2018-11-19   4871.490234     -0.133732


In [46]:
worst_days = data.nsmallest(10, "daily_return")[["Date","Close","daily_return"]]

In [47]:
print(worst_days)

           Date         Close  daily_return
1166 2020-03-12   4970.788086     -0.371695
256  2017-09-14   3154.949951     -0.187411
380  2018-01-16  11490.500000     -0.168548
1989 2022-06-13  22487.388672     -0.159747
400  2018-02-05   6955.270020     -0.159688
2138 2022-11-09  15880.780273     -0.143490
10   2017-01-11    777.757019     -0.143136
907  2019-06-27  11182.806641     -0.140857
1599 2021-05-19  37002.441406     -0.137661
687  2018-11-19   4871.490234     -0.133732


In [48]:
px.bar(worst_days,
       x="Date",
       y=["daily_return"],
       title = "Bitcoin- Worst Crash Days In History"
       )

In [49]:
worst_days["crash_label"] = "BTC | " + worst_days["Date"].dt.strftime("%Y-%m-%d")

In [50]:
fig = px.bar(worst_days,
       x="crash_label",
       y=["daily_return"],
       title = "Bitcoin- Worst Crash Days In History",
       labels={
           "daily_return" : "Daily return(Loss)",
           "Crash_day" : "Crash_Date"
       })
fig.add_hline(y=0)
fig.show()


In [51]:
data.columns

Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'daily_return',
       'volatility_30d', 'volatility_90d', 'cumulative_return',
       'investment_value'],
      dtype='str')

In [52]:
data["price_change"] = data["Close"] - data["Open"]

In [53]:
data["Volume"]

0         147775008
1         222184992
2         185168000
3         344945984
4         510199008
           ...     
3281    42455674908
3282    13741199310
3283    15156557929
3284    48411625849
3285    35586356225
Name: Volume, Length: 3286, dtype: int64

In [54]:
data["Volume_change"] = data["Volume"].pct_change()

In [55]:
def classify_volume_price(row):
    if row["price_change"] > 0 and row["Volume_change"] > 0:
        return "Strong Bullish"
    elif row["price_change"] < 0 and row["Volume_change"] > 0:
        return "Strong Bearish"
    elif row["price_change"] > 0 and row["Volume_change"] < 0:
        return "Weak Bullish"
    else:
        return "Weak Bearish"

In [56]:
data["volume_price_signal"] = data.apply(classify_volume_price, axis = 1)

In [57]:
data["volume_price_signal"].value_counts()

volume_price_signal
Strong Bullish    865
Weak Bullish      860
Weak Bearish      824
Strong Bearish    737
Name: count, dtype: int64

In [58]:
data.head(1)

,Date,Close,High,Low,Open,Volume,daily_return,volatility_30d,volatility_90d,cumulative_return,investment_value,price_change,Volume_change,volume_price_signal
0,2017-01-01,998.325012,1003.080017,958.698975,963.65802,147775008,NaN,NaN,NaN,NaN,NaN,34.666992,NaN,Weak Bearish


In [59]:
data["Year"] = data["Date"].dt.year

In [60]:
yearly_returns = data.groupby("Year")["daily_return"].apply( lambda x : (1 + x).prod() - 1)

In [61]:
yearly_returns.idxmax()

np.int32(2017)

In [62]:
yearly_returns.idxmin()

np.int32(2018)

In [63]:
yearly_df = yearly_returns.reset_index()

In [64]:
yearly_df.columns = ["Year", "Return (%)"]

In [65]:
print(yearly_df)

   Year  Return (%)
0  2017   13.180152
1  2018   -0.735618
2  2019    0.922034
3  2020    3.031601
4  2021    0.596679
5  2022   -0.642652
6  2023    1.554174
7  2024    1.210547
8  2025   -0.053507


In [66]:
px.bar(yearly_df,
       x = "Year",
       y = "Return (%)",
       title = "Bitcoin- Yearly Returns Best vs Worst Years"
       )